# v003 · Trích feature (AV-HuBERT Large, crop miệng, XLS-R)

Đầu vào: dataset đã finalize của các part (`datasets/vn-av-df-data/<part>`). Đầu ra: `cache/vn-av-df-data/{avhubert,mouth,xlsr}`.
Hết giờ phiên thì Save Version, phiên sau attach `PREVIOUS_CACHE` để trích tiếp (cache đã có được bỏ qua).

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json, shutil
SESSION_START = time.time()
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa thành repository của bạn.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
WORKERS = Path("/kaggle/temp/vn-av-df-workers")  # Môi trường worker: ngoài Output 20 GB.
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT / "data_pipeline"),
                "-e", str(ROOT) + "[training]"], check=True)
import settings
settings.bootstrap()

In [ ]:
# W&B: log quá trình train (trích feature, train, test, báo cáo) làm bằng chứng. Project PRIVATE.
# Thêm secret WANDB_API_KEY: Add-ons → Secrets. Không đẩy ảnh / video người thật.
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-v003"
if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_DIR"] = "/kaggle/temp"  # file tạm W&B ở ngoài Output
    os.makedirs(os.environ["WANDB_DIR"], exist_ok=True)
WANDB = {"project": WANDB_PROJECT} if USE_WANDB else None

In [ ]:
DATASET_PARTS = [Path("/kaggle/input/datasets/YOUR_USER/vn-av-df-data-part1")]
PREVIOUS_CACHE = None  # Path("/kaggle/input/<phiên trước>/vn-av-df-forensics/cache/vn-av-df-data")
ENCODERS = ["avhubert", "mouth", "xlsr"]
import torch
from vn_av_df.dataset import part_selection
settings.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg = settings.config()
root, parts = part_selection(DATASET_PARTS)
cfg["dataset"], cfg["dataset_parts"] = str(root), parts
cfg["prepare_devices"] = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or [None]
cfg["wandb"] = WANDB
if PREVIOUS_CACHE and not Path(cfg["cache"]).exists():
    shutil.copytree(PREVIOUS_CACHE, cfg["cache"])

In [ ]:
from vn_av_df.cli import setup
print(setup(cfg, generators=[], encoders=True))  # AV-HuBERT Large + XLS-R
if "avhubert" in ENCODERS:
    subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "avhubert",
                    "--root", str(ROOT), "--env-root", str(WORKERS)], check=True)
    cfg["encoders"]["avhubert"]["python"] = str(WORKERS / ".venv-avhubert/bin/python")

In [ ]:
from vn_av_df.training.prepare import prepare
print(prepare(cfg, ENCODERS))
from vn_av_data.common.runtime import prune_output
print("Đã xoá khỏi Output:", len(prune_output(ROOT, [str(Path(cfg["cache"]).relative_to(ROOT))])), "mục")